<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_26_practicum_dp_greedy/note_lesson_26_scheduling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚕 Урок 26 — Практикум П5: розклад водія (greedy + динамічне програмування)

Водій «Смачно + Таксі» обирає замовлення на зміну. Дві мети на тих самих даних:

1. **найбільше замовлень** — жадібний алгоритм за найранішим кінцем;
2. **найбільший заробіток** — динамічне програмування «взяти або пропустити».

| Крок | Що робимо |
|---|---|
| 0 | дані й модель |
| 1–2 | сумісність і жадібний розклад (вправи 1–2) |
| 3 | зміна мети: контрприклад (вправа 3) |
| 4–6 | `p(i)`, таблиця ДП, відновлення розкладу (вправи 4–6) |
| 7 | перевірка перебором (вправа 7) |
| 8 | розширення: `bisect` (вправа 8) |

**Як працювати:** виконуй клітинки **зверху вниз**; вправи спираються одна на одну. Перед **🔮 Прогнозом** спершу відповідай сам. Теорія, таблиці й покрокові схеми — у книзі: [Урок 26](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_26/).

---
## 0. Дані й модель

| Замовлення | Час | Заробіток |
|---|---|---:|
| А | 09:00–10:00 | 200 грн |
| Б | 10:00–11:00 | 200 грн |
| В | 09:00–11:00 | 550 грн |
| Г | 11:00–12:00 | 200 грн |

Спрощення: інтервали фіксовані; одночасно — одне замовлення; кінець о 10:00 дозволяє почати о 10:00; дорогу й витрати не враховуємо; максимізуємо винагороду. Час — цілі години.

In [ ]:
from typing import NamedTuple


class Order(NamedTuple):
    name: str
    start: int
    end: int
    reward: int


orders = [
    Order("А", 9, 10, 200),
    Order("Б", 10, 11, 200),
    Order("В", 9, 11, 550),
    Order("Г", 11, 12, 200),
]


def names(schedule):
    return [o.name for o in schedule]


def total(schedule):
    return sum(o.reward for o in schedule)


print(names(orders), total(orders))

**🔮 Прогноз:** яку найбільшу **кількість** замовлень може виконати водій і скільки найбільше **заробити**? Запиши обидва розклади.

<details>
<summary>Відповідь</summary>

3 замовлення (А → Б → Г, 600 грн) і 750 грн (В → Г). Різні розклади для різних цілей — про це весь урок.

</details>

---
## 1. Сумісність

### Вправа 1. `compatible(a, b)`

Два замовлення сумісні, якщо одне закінчується **не пізніше**, ніж починається інше (у будь-якому порядку).

In [ ]:
def compatible(a, b):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return a.end <= b.start or b.end <= a.start
    # END SOLUTION


А, Б, В, Г = orders
assert compatible(А, Б) and compatible(Б, А)          # дотичні межі — можна
assert not compatible(А, В) and not compatible(В, Б)  # перетин
assert compatible(В, Г)
assert not compatible(В, В)
print("✅ Вправа 1 пройдена")

---
## 2. Мета 1: найбільше замовлень (greedy)

Правило: відсортуй за **часом завершення**, бери замовлення, якщо воно починається не раніше, ніж закінчилось останнє взяте. Ніколи не повертайся до прийнятих рішень.

**🔮 Прогноз:** у якому порядку `sorted(orders, key=lambda o: o.end)` поставить Б і В (обидва закінчуються об 11)? Які замовлення візьме жадібний алгоритм?

<details>
<summary>Відповідь</summary>

Б, потім В: `sorted` **стабільний** і зберігає початковий порядок рівних. Жадібний візьме А (кінець 10), Б (10 ≥ 10), пропустить В (9 < 11), візьме Г (11 ≥ 11).

</details>

### Вправа 2. `max_orders(orders)`

Поверни список узятих замовлень у порядку взяття.

In [ ]:
def max_orders(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    chosen = []
    last_end = None
    for order in sorted(orders, key=lambda o: o.end):
        if last_end is None or order.start >= last_end:
            chosen.append(order)
            last_end = order.end
    return chosen
    # END SOLUTION


print(names(max_orders(orders)), total(max_orders(orders)))
assert names(max_orders(orders)) == ["А", "Б", "Г"]
assert max_orders([]) == []
early = [Order("Л", 8, 12, 0), Order("М", 9, 10, 0), Order("Н", 10, 11, 0)]
assert names(max_orders(early)) == ["М", "Н"]
print("✅ Вправа 2 пройдена")

---
## 3. Зміна мети: гроші

Жадібний розклад дає 3 замовлення, але лише 600 грн — а В + Г дають 750. Правило «найраніший кінець» не дивиться на винагороду.

### Вправа 3. Свій контрприклад

Придумай **свою** зміну `my_shift` з 3–5 замовлень, де жадібний за кількістю розклад заробляє **менше**, ніж найкращий. Найкращий заробіток поки порахуй вручну й запиши в `my_best`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
my_shift = [Order("Д", 9, 13, 500), Order("Е", 9, 12, 100), Order("Ж", 12, 13, 100)]
my_best = 500
# END SOLUTION

greedy_money = total(max_orders(my_shift))
print("greedy:", names(max_orders(my_shift)), greedy_money, "грн; найкраще:", my_best, "грн")
assert 3 <= len(my_shift) <= 5
assert greedy_money < my_best
print("✅ Вправа 3 пройдена (у вправі 7 перебір перевірить, що my_best справді найкраще)")

---
## 4. ДП: позначення і `p(i)`

Відсортуй замовлення за `end` і пронумеруй з **одиниці**: замовлення `i` — це `orders[i - 1]`.

- `dp[i]` — найбільший заробіток, якщо дозволено брати лише перші `i` замовлень; `dp[0] = 0`;
- `p(i)` — скільки перших замовлень закінчуються не пізніше за **початок** замовлення `i`.

Перехід: `dp[i] = max(dp[i - 1], reward(i) + dp[p(i)])` — пропустити або взяти.

**🔮 Прогноз:** чому дорівнюють `p(1)`, `p(2)`, `p(3)`, `p(4)` для порядку А, Б, В, Г?

<details>
<summary>Відповідь</summary>

`[0, 1, 0, 3]`: до 9 не закінчилось нічого; до 10 — А; до 9 — нічого; до 11 — А, Б, В.

</details>

### Вправа 4. `prev_compatible(orders)`

Замовлення **вже відсортовані** за `end`. Поверни список `p` довжини `n + 1` (`p[0]` не використовується, хай буде `0`). Найпростіший спосіб: для кожного `i` йти назад від `i - 1` до `1` і зупинитись на першому `j`, що закінчився до початку `i`.

In [ ]:
def prev_compatible(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    p = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        start = orders[i - 1].start
        for j in range(i - 1, 0, -1):
            if orders[j - 1].end <= start:
                p[i] = j
                break
    return p
    # END SOLUTION


by_end = sorted(orders, key=lambda o: o.end)
print(names(by_end), prev_compatible(by_end))
assert prev_compatible(by_end) == [0, 0, 1, 0, 3]
assert prev_compatible([]) == [0]
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** заповни таблицю: `dp[0]` … `dp[4]`. Для кожного рядка запиши «пропустити» і «взяти».

<details>
<summary>Відповідь</summary>

| i | замовлення | p | пропустити | взяти | dp |
|---|---|---|---|---|---|
| 0 | — | — | — | — | 0 |
| 1 | А 200 | 0 | 0 | 200 | 200 |
| 2 | Б 200 | 1 | 200 | 400 | 400 |
| 3 | В 550 | 0 | 400 | 550 | 550 |
| 4 | Г 200 | 3 | 550 | 750 | 750 |

</details>

### Вправа 5. `max_earnings(orders)`

Сортування → `p` → список `dp` → цикл від `1` до `n`. Поверни `dp[n]`. Порядок вхідних замовлень може бути будь-який.

In [ ]:
def max_earnings(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        dp[i] = max(dp[i - 1], orders[i - 1].reward + dp[p[i]])
    return dp[-1]
    # END SOLUTION


print(max_earnings(orders))
assert max_earnings(orders) == 750
assert max_earnings(list(reversed(orders))) == 750
assert max_earnings([]) == 0
evening = [Order("Д", 9, 13, 500), Order("Е", 9, 11, 300), Order("Ж", 11, 13, 300)]
assert max_earnings(evening) == 600
print("✅ Вправа 5 пройдена")

---
## 6. Який розклад дав максимум

### Вправа 6. `best_schedule(orders)`

Заповни `dp`, як у вправі 5, а потім іди **з кінця**: якщо `reward(i) + dp[p(i)] > dp[i - 1]` — замовлення `i` в розкладі, стрибай у `p(i)`; інакше — у `i - 1`. Поверни розклад у порядку часу.

In [ ]:
def best_schedule(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        dp[i] = max(dp[i - 1], orders[i - 1].reward + dp[p[i]])
    chosen = []
    i = len(orders)
    while i > 0:
        if orders[i - 1].reward + dp[p[i]] > dp[i - 1]:
            chosen.append(orders[i - 1])
            i = p[i]
        else:
            i -= 1
    return list(reversed(chosen))
    # END SOLUTION


print(names(best_schedule(orders)), total(best_schedule(orders)))
assert names(best_schedule(orders)) == ["В", "Г"]
assert names(best_schedule(evening)) == ["Е", "Ж"]
assert best_schedule([]) == []
print("✅ Вправа 6 пройдена")

---
## 7. Перевірка перебором

Повний перебір усіх `2ⁿ` підмножин — повільний, але очевидно правильний суддя для малих `n` (урок 25).

### Вправа 7. `brute_force(orders)`

Поверни найбільший заробіток серед усіх **сумісних** підмножин. Підказка: `itertools.combinations(orders, size)` для кожного `size` від 0 до `n`, а сумісність усіх пар — `combinations(combo, 2)`.

In [ ]:
from itertools import combinations
import random


def brute_force(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    best = 0
    for size in range(len(orders) + 1):
        for combo in combinations(orders, size):
            if all(compatible(a, b) for a, b in combinations(combo, 2)):
                best = max(best, total(combo))
    return best
    # END SOLUTION


cases = {
    "порожній список": [],
    "усі сумісні": [Order("x", 9, 10, 100), Order("y", 10, 11, 150), Order("z", 11, 12, 50)],
    "усі перетинаються": [Order("x", 9, 12, 100), Order("y", 10, 13, 150), Order("z", 11, 14, 50)],
    "дотичні межі": orders,
    "однаковий кінець": [Order("x", 9, 11, 100), Order("y", 10, 11, 150)],
    "твій контрприклад": my_shift,
}
for name, data in cases.items():
    print(f"{name:18} ДП = {max_earnings(data):4}  перебір = {brute_force(data):4}")
    assert max_earnings(data) == brute_force(data)
assert brute_force(my_shift) == my_best, "my_best у вправі 3 — не найкращий заробіток"

random.seed(26)
for trial in range(300):
    data = []
    for k in range(random.randint(0, 8)):
        start = random.randint(8, 20)
        data.append(Order(str(k), start, start + random.randint(1, 4), random.randint(1, 10) * 50))
    assert max_earnings(data) == brute_force(data), data
    assert total(best_schedule(data)) == max_earnings(data), data
print("✅ Вправа 7 пройдена: 300 випадкових змін без розбіжностей")

---
## 8. Розширення: `p(i)` через `bisect`

`prev_compatible` у гіршому разі — `O(n²)`. Але кінці відсортовані, і `p(i)` — це кількість кінців `<= start` серед перших `i - 1`: `bisect.bisect_right(ends, start, 0, i - 1)` (урок 11).

### Вправа 8. `max_earnings_fast(orders)`

Те саме ДП без вкладених циклів.

In [ ]:
import bisect


def max_earnings_fast(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    orders = sorted(orders, key=lambda o: o.end)
    ends = [o.end for o in orders]
    dp = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        p = bisect.bisect_right(ends, orders[i - 1].start, 0, i - 1)
        dp[i] = max(dp[i - 1], orders[i - 1].reward + dp[p])
    return dp[-1]
    # END SOLUTION


shift = [
    Order("А", 9, 10, 200), Order("Б", 10, 11, 200), Order("В", 9, 11, 550),
    Order("Г", 11, 12, 200), Order("Ґ", 12, 14, 380), Order("Д", 11, 13, 300),
    Order("Е", 13, 15, 250), Order("Є", 14, 16, 420), Order("Ж", 15, 17, 260),
    Order("З", 16, 18, 330),
]
print(max_earnings_fast(orders), max_earnings_fast(shift))
assert max_earnings_fast(orders) == 750
assert max_earnings_fast(shift) == 1880
assert max_earnings_fast([]) == 0
random.seed(8)
for trial in range(300):
    data = [Order(str(k), s, s + random.randint(1, 4), random.randint(1, 10) * 50)
            for k, s in enumerate(random.randint(8, 20) for _ in range(random.randint(0, 12)))]
    assert max_earnings_fast(data) == max_earnings(data), data
print("✅ Вправа 8 пройдена")

---
## Самоперевірка

1. Чому правило «найраніший кінець» оптимальне для кількості, але не для заробітку?
2. Що означає `dp[i]` і навіщо нумерація з одиниці?
3. Чому для ДП замовлення сортують саме за `end`?
4. Як з таблиці `dp` дістати сам розклад?
5. Навіщо повільний `brute_force`, якщо є швидке ДП?

<details>
<summary>Відповіді</summary>

1. Для кількості кожне замовлення важить +1, і раніше звільнитись завжди не гірше (аргумент обміну). Для заробітку одне дороге замовлення може бути вигідніше за кілька дешевих — правило цього не бачить (600 проти 750).
2. Найбільший заробіток серед перших `i` замовлень. Нумерація з одиниці дає базу `dp[0] = 0` — «жодного замовлення».
3. Тоді всі замовлення, сумісні з `i` і розміщені перед ним, — це префікс довжини `p(i)`, а найкраще для нього вже пораховано: `dp[p(i)]`.
4. Іти з кінця: якщо «взяти» перемогло — замовлення в розкладі, стрибок у `p(i)`; інакше — `i - 1`.
5. Щоб перевірити ДП на малих граничних і випадкових даних: перебір очевидно правильний.

</details>

## Далі

- Теорія, покрокові схеми, аргумент обміну, мемоізація проти табуляції: [Урок 26](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_26/).
- Додатковий ноутбук про модуль `re` і жадібні квантифікатори: `re_lesson_26_greedy_regex.ipynb` у цій самій папці.
- **Урок 27** — потоки, multiprocessing, asyncio: вступ.